In [1]:
import pandas as pd
import numpy as np
import re

# Load the dataset with the specified delimiter
file_path = r"C:\Users\anouk\OneDrive - UvA\Documenten\GitHub\the good one\tweede-kamer\data\speeches2014-2024\party_affilliations_fixed\speeches_2014-2024_speakers_parties_fixed_with_dob.csv"
df = pd.read_csv(file_path, sep=';')

# Check if 'title' and 'url' columns exist
if 'title' in df.columns and 'url' in df.columns:
    # Extract debate ID from URL using regex pattern
    df['debate_id'] = df['url'].apply(lambda x: re.search(r'([^/]+)$', str(x)).group(1) if pd.notna(x) and re.search(r'([^/]+)$', str(x)) else None)
    
    # Create a normalized title column by removing all digits and standardizing whitespace
    df['normalized_title'] = df['title'].astype(str).apply(
        lambda x: re.sub(r'\d+', '', x).strip()  # Remove all digits and trim whitespace
    )
    # Further normalize by removing extra spaces
    df['normalized_title'] = df['normalized_title'].apply(
        lambda x: re.sub(r'\s+', ' ', x).strip()  # Replace multiple spaces with a single space
    )
    
    # Calculate counts based on normalized titles
    normalized_title_speech_counts = df.groupby('normalized_title')['speech_id'].count()
    
    # Count unique debate IDs per normalized title
    normalized_title_debate_counts = df.groupby('normalized_title')['debate_id'].nunique()
    
    # Create a DataFrame to hold all the normalized counts
    normalized_title_stats = pd.DataFrame({
        'speech_count': normalized_title_speech_counts,
        'debate_count': normalized_title_debate_counts,
        'original_titles': df.groupby('normalized_title')['title'].unique().apply(lambda x: list(x))
    })
    
    # Remove rows with empty normalized titles
    normalized_title_stats = normalized_title_stats[normalized_title_stats.index != '']
    
    # Print the number of unique normalized titles
    print(f"Number of unique normalized debate titles (ignoring numbers): {len(normalized_title_stats)}")
    print(f"Original number of unique titles: {df['title'].nunique()}")
    print(f"Reduction: {df['title'].nunique() - len(normalized_title_stats)} titles merged")
    
    # Sort by unique debate count (most unique debates first)
    print("\n=== NORMALIZED TITLES SORTED BY NUMBER OF UNIQUE DEBATES (DESCENDING) ===")
    print("="*105)
    print("{:<60} | {:<15} | {:<15} | {:<15}".format("Normalized Title", "Unique Debates", "Speech Count", "Original Title Count"))
    print("-"*105)
    
    sorted_by_debate = normalized_title_stats.sort_values('debate_count', ascending=False)
    for title, row in sorted_by_debate.iterrows():
        # Print the normalized title with counts and number of original variations
        print("{:<60} | {:<15} | {:<15} | {:<15}".format(
            title[:58] + ".." if len(title) > 60 else title, 
            row['debate_count'],
            row['speech_count'],
            len(row['original_titles'])
        ))
    
    # Print some statistics
    print("\nSummary Statistics:")
    print(f"Total speeches: {len(df)}")
    print(f"Total unique debate IDs: {df['debate_id'].nunique()}")
    print(f"Original unique titles: {df['title'].nunique()}")
    print(f"Normalized unique titles: {len(normalized_title_stats)}")
    
    # Show examples of top normalized debates with all their original titles
    print("\nTop 5 most frequent normalized debate topics (by unique debate count):")
    for i, (title, row) in enumerate(sorted_by_debate.head(5).iterrows(), 1):
        print(f"\n{i}. \"{title}\"")
        print(f"   - Unique debates: {row['debate_count']}")
        print(f"   - Total speeches: {row['speech_count']}")
        print(f"   - Original title variations ({len(row['original_titles'])}):")
        for j, orig_title in enumerate(sorted(row['original_titles'])[:10], 1):
            print(f"     {j}. \"{orig_title}\"")
        if len(row['original_titles']) > 10:
            print(f"     ... and {len(row['original_titles'])-10} more variations")
    
else:
    missing_cols = []
    if 'title' not in df.columns:
        missing_cols.append('title')
    if 'url' not in df.columns:
        missing_cols.append('url')
    
    print(f"The following columns are missing: {', '.join(missing_cols)}")
    print("Available columns:", df.columns.tolist())

Number of unique normalized debate titles (ignoring numbers): 8645
Original number of unique titles: 12445
Reduction: 3800 titles merged

=== NORMALIZED TITLES SORTED BY NUMBER OF UNIQUE DEBATES (DESCENDING) ===
Normalized Title                                             | Unique Debates  | Speech Count    | Original Title Count
---------------------------------------------------------------------------------------------------------
Mededelingen                                                 | 1809            | 2262            | 57             
Regeling van werkzaamheden                                   | 1080            | 86787           | 53             
Regeling van werkzaamheden (stemmingen)                      | 343             | 2945            | 29             
Vragenuur                                                    | 68              | 2054            | 3              
Ontwikkelingen rondom het coronavirus                        | 49              | 22307           | 15 

In [2]:
import pandas as pd
import numpy as np
import re

# Load the dataset with the specified delimiter
file_path = r"C:\Users\anouk\OneDrive - UvA\Documenten\GitHub\the good one\tweede-kamer\data\speeches2014-2024\party_affilliations_fixed\speeches_2014-2024_speakers_parties_fixed_with_dob.csv"
df = pd.read_csv(file_path, sep=';')

# Check if 'title' and 'url' columns exist
if 'title' in df.columns and 'url' in df.columns:
    # Extract debate ID from URL using regex pattern
    df['debate_id'] = df['url'].apply(lambda x: re.search(r'([^/]+)$', str(x)).group(1) if pd.notna(x) and re.search(r'([^/]+)$', str(x)) else None)
    
    # Create a normalized title column by removing all digits and standardizing whitespace
    df['normalized_title'] = df['title'].astype(str).apply(
        lambda x: re.sub(r'\d+', '', x).strip()  # Remove all digits and trim whitespace
    )
    # Further normalize by removing extra spaces
    df['normalized_title'] = df['normalized_title'].apply(
        lambda x: re.sub(r'\s+', ' ', x).strip()  # Replace multiple spaces with a single space
    )
    
    # Filter for debates starting with "begroting" (case insensitive)
    begroting_filter = df['normalized_title'].str.lower().str.startswith('begroting')
    begroting_df = df[begroting_filter]
    
    # Calculate counts for begroting debates
    begroting_stats = pd.DataFrame({
        'speech_count': begroting_df.groupby('normalized_title')['speech_id'].count(),
        'debate_count': begroting_df.groupby('normalized_title')['debate_id'].nunique(),
        'original_titles': begroting_df.groupby('normalized_title')['title'].unique().apply(lambda x: list(x))
    })
    
    # Sort by unique debate count (most unique debates first)
    sorted_begroting = begroting_stats.sort_values('debate_count', ascending=False)
    
    # Print summary statistics
    total_begroting_debates = begroting_df['debate_id'].nunique()
    total_begroting_speeches = len(begroting_df)
    total_begroting_normalized_titles = len(sorted_begroting)
    
    print(f"=== BEGROTING DEBATE STATISTICS ===")
    print(f"Total unique normalized 'begroting' titles: {total_begroting_normalized_titles}")
    print(f"Total unique 'begroting' debate IDs: {total_begroting_debates}")
    print(f"Total 'begroting' speeches: {total_begroting_speeches}")
    print(f"Percentage of all debates: {(total_begroting_debates / df['debate_id'].nunique()) * 100:.2f}%")
    print(f"Percentage of all speeches: {(total_begroting_speeches / len(df)) * 100:.2f}%")
    
    print("\n=== ALL NORMALIZED 'BEGROTING' TITLES SORTED BY DEBATE COUNT ===")
    print("="*105)
    print("{:<60} | {:<15} | {:<15} | {:<15}".format("Normalized Title", "Unique Debates", "Speech Count", "Original Title Count"))
    print("-"*105)
    
    for title, row in sorted_begroting.iterrows():
        print("{:<60} | {:<15} | {:<15} | {:<15}".format(
            title[:58] + ".." if len(title) > 60 else title, 
            row['debate_count'],
            row['speech_count'],
            len(row['original_titles'])
        ))
    
    # Show sample of original titles for each normalized title
    print("\n=== SAMPLE OF ORIGINAL TITLES FOR EACH NORMALIZED 'BEGROTING' TITLE ===")
    for title, row in sorted_begroting.iterrows():
        print(f"\n\"{title}\"")
        print(f"   - Unique debates: {row['debate_count']}")
        print(f"   - Total speeches: {row['speech_count']}")
        print(f"   - Original title variations ({len(row['original_titles'])}):")
        sample_size = min(5, len(row['original_titles']))  # Show up to 5 examples
        for j, orig_title in enumerate(sorted(row['original_titles'])[:sample_size], 1):
            print(f"     {j}. \"{orig_title}\"")
        if len(row['original_titles']) > sample_size:
            print(f"     ... and {len(row['original_titles']) - sample_size} more variations")
    
else:
    missing_cols = []
    if 'title' not in df.columns:
        missing_cols.append('title')
    if 'url' not in df.columns:
        missing_cols.append('url')
    
    print(f"The following columns are missing: {', '.join(missing_cols)}")
    print("Available columns:", df.columns.tolist())

=== BEGROTING DEBATE STATISTICS ===
Total unique normalized 'begroting' titles: 50
Total unique 'begroting' debate IDs: 371
Total 'begroting' speeches: 94532
Percentage of all debates: 2.09%
Percentage of all speeches: 11.93%

=== ALL NORMALIZED 'BEGROTING' TITLES SORTED BY DEBATE COUNT ===
Normalized Title                                             | Unique Debates  | Speech Count    | Original Title Count
---------------------------------------------------------------------------------------------------------
Begroting Sociale Zaken en Werkgelegenheid                   | 30              | 8814            | 27             
Begroting Buitenlandse Zaken                                 | 30              | 7057            | 25             
Begroting Volksgezondheid, Welzijn en Sport                  | 29              | 8766            | 19             
Begroting Onderwijs, Cultuur en Wetenschap                   | 29              | 8851            | 24             
Begroting Defensie    

In [3]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

# Load the dataset with the specified delimiter
file_path = r"C:\Users\anouk\OneDrive - UvA\Documenten\GitHub\the good one\tweede-kamer\data\speeches2014-2024\party_affilliations_fixed\speeches_2014-2024_speakers_parties_fixed_with_dob.csv"
df = pd.read_csv(file_path, sep=';')

# Check if required columns exist
if all(col in df.columns for col in ['title', 'url', 'date']):
    # Extract debate ID from URL
    df['debate_id'] = df['url'].apply(lambda x: re.search(r'([^/]+)$', str(x)).group(1) if pd.notna(x) and re.search(r'([^/]+)$', str(x)) else None)
    
    # Create a normalized title column by removing all digits and standardizing whitespace
    df['normalized_title'] = df['title'].astype(str).apply(
        lambda x: re.sub(r'\d+', '', x).strip()  # Remove all digits
    )
    df['normalized_title'] = df['normalized_title'].apply(
        lambda x: re.sub(r'\s+', ' ', x).strip()  # Standardize spaces
    )
    
    # Convert date column to datetime and extract year
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    
    # Get the range of years in the dataset
    years = sorted(df['year'].unique())
    
    # Filter for debates starting with "begroting" (case insensitive)
    begroting_filter = df['normalized_title'].str.lower().str.startswith('begroting')
    begroting_df = df[begroting_filter]
    
    # Create a dataframe with unique debate IDs for begroting debates
    unique_begroting_debates = begroting_df.drop_duplicates(['normalized_title', 'debate_id', 'year'])
    
    # Count total debates per normalized title
    normalized_title_debate_counts = unique_begroting_debates.groupby('normalized_title')['debate_id'].count()
    
    # Count speeches per normalized title
    normalized_title_speech_counts = begroting_df.groupby('normalized_title')['speech_id'].count()
    
    # Get original title variations
    normalized_title_variations = begroting_df.groupby('normalized_title')['title'].unique().apply(lambda x: list(x))
    
    # Create counts by year for each normalized title
    yearly_counts = {}
    for year in years:
        year_filter = unique_begroting_debates['year'] == year
        yearly_counts[year] = unique_begroting_debates[year_filter].groupby('normalized_title')['debate_id'].count()
    
    # Create a DataFrame to hold all statistics
    stats_columns = ['debate_count', 'speech_count', 'original_title_count']
    stats_columns.extend([f'debates_{year}' for year in years])
    
    begroting_stats = pd.DataFrame(index=normalized_title_debate_counts.index, columns=stats_columns)
    begroting_stats['debate_count'] = normalized_title_debate_counts
    begroting_stats['speech_count'] = normalized_title_speech_counts
    begroting_stats['original_title_count'] = normalized_title_variations.apply(len)
    
    # Fill in yearly counts
    for year in years:
        begroting_stats[f'debates_{year}'] = yearly_counts[year]
    
    # Fill NaN values with 0 for year columns
    for year in years:
        begroting_stats[f'debates_{year}'] = begroting_stats[f'debates_{year}'].fillna(0).astype(int)
    
    # Calculate what percentage of all debates each normalized title represents
    total_begroting_debates = unique_begroting_debates['debate_id'].nunique()
    begroting_stats['debate_percentage'] = (begroting_stats['debate_count'] / total_begroting_debates * 100).round(2)
    
    # Sort by total debate count (descending)
    sorted_begroting = begroting_stats.sort_values('debate_count', ascending=False)
    
    # Print summary statistics
    print(f"=== BEGROTING DEBATE STATISTICS ===")
    print(f"Analysis of 'begroting' debates from {min(years)} to {max(years)}")
    print(f"Total unique normalized 'begroting' titles: {len(begroting_stats)}")
    print(f"Total unique 'begroting' debate IDs: {total_begroting_debates}")
    print(f"Total 'begroting' speeches: {len(begroting_df)}")
    print(f"Percentage of all debates: {(total_begroting_debates / df['debate_id'].nunique()) * 100:.2f}%")
    print(f"Percentage of all speeches: {(len(begroting_df) / len(df)) * 100:.2f}%")
    
    # Divide into two categories:
    # 1. Topics that have at least one debate each year
    # 2. Topics that don't have debates in every year
    
    # Identify titles with at least one debate in each year
    titles_with_debates_each_year = []
    titles_without_debates_each_year = []
    
    for title, row in sorted_begroting.iterrows():
        has_debate_each_year = True
        for year in years:
            if row[f'debates_{year}'] == 0:
                has_debate_each_year = False
                break
                
        if has_debate_each_year:
            titles_with_debates_each_year.append(title)
        else:
            titles_without_debates_each_year.append(title)
    
    # Create separate DataFrames
    consistent_topics = sorted_begroting.loc[titles_with_debates_each_year]
    inconsistent_topics = sorted_begroting.loc[titles_without_debates_each_year]
    
    # Print the titles with debates in every year
    print("\n=== NORMALIZED 'BEGROTING' TITLES WITH AT LEAST ONE DEBATE EACH YEAR ===")
    print("="*130)
    
    # Create a header with year columns
    header_format = "{:<40} | {:<8} | {:<8} | {:<8} | " + " | ".join(["{:<8}"] * len(years))
    print(header_format.format("Normalized Title", "Debates", "Speeches", "Orig. #", *[str(year) for year in years]))
    print("-"*130)
    
    # Print each row with yearly counts
    row_format = "{:<40} | {:<8} | {:<8} | {:<8} | " + " | ".join(["{:<8}"] * len(years))
    for title, row in consistent_topics.iterrows():
        title_display = title[:38] + ".." if len(title) > 40 else title
        yearly_values = [row[f'debates_{year}'] for year in years]
        print(row_format.format(
            title_display, 
            row['debate_count'], 
            row['speech_count'],
            row['original_title_count'],
            *yearly_values
        ))
    
    print(f"\nTotal topics with debates in every year: {len(consistent_topics)}")
    
    # Print the titles without debates in every year
    print("\n=== NORMALIZED 'BEGROTING' TITLES WITH GAPS IN YEARLY DEBATES ===")
    print("="*130)
    
    print(header_format.format("Normalized Title", "Debates", "Speeches", "Orig. #", *[str(year) for year in years]))
    print("-"*130)
    
    for title, row in inconsistent_topics.iterrows():
        title_display = title[:38] + ".." if len(title) > 40 else title
        yearly_values = [row[f'debates_{year}'] for year in years]
        print(row_format.format(
            title_display, 
            row['debate_count'], 
            row['speech_count'],
            row['original_title_count'],
            *yearly_values
        ))
    
    print(f"\nTotal topics with gaps in yearly debates: {len(inconsistent_topics)}")
    
    # FIX: Only show a few examples of original titles for each normalized title to avoid the error
    print("\n=== SAMPLE OF ORIGINAL TITLES FOR EACH NORMALIZED 'BEGROTING' TITLE ===")
    for title, row in sorted_begroting.iterrows():
        print(f"\n\"{title}\"")
        print(f"   - Unique debates: {row['debate_count']}")
        print(f"   - Total speeches: {row['speech_count']}")
        print(f"   - Original title variations ({row['original_title_count']}):")
        
        # FIX: Make sure sample_size is an integer
        sample_size = min(5, int(row['original_title_count']))
        
        # Get the list of original titles for this normalized title
        orig_titles = normalized_title_variations.get(title, [])
        
        # Take only a few examples
        for j, orig_title in enumerate(sorted(orig_titles)[:sample_size], 1):
            print(f"     {j}. \"{orig_title}\"")
        
        if len(orig_titles) > sample_size:
            print(f"     ... and {len(orig_titles) - sample_size} more variations")
    
    # Save the results to separate CSV files
   # output_path = r"C:\Users\anouk\OneDrive - UvA\Documenten\GitHub\the good one\tweede-kamer\data\speeches2014-2024\begroting_debates_by_year.csv"
   # sorted_begroting.to_csv(output_path)
    
   # consistent_path = r"C:\Users\anouk\OneDrive - UvA\Documenten\GitHub\the good one\tweede-kamer\data\speeches2014-2024\begroting_debates_consistent.csv"
   # inconsistent_path = r"C:\Users\anouk\OneDrive - UvA\Documenten\GitHub\the good one\tweede-kamer\data\speeches2014-2024\begroting_debates_inconsistent.csv"
    
   # consistent_topics.to_csv(consistent_path)
   # inconsistent_topics.to_csv(inconsistent_path)
    
   # print(f"\nDetailed results saved to:")
   # print(f"- All begroting topics: {output_path}")
   # print(f"- Topics with debates every year: {consistent_path}")
   # print(f"- Topics with gaps in yearly debates: {inconsistent_path}")
    
else:
    missing_cols = []
    for col in ['title', 'url', 'date']:
        if col not in df.columns:
            missing_cols.append(col)
    
    print(f"The following required columns are missing: {', '.join(missing_cols)}")
    print("Available columns:", df.columns.tolist())

=== BEGROTING DEBATE STATISTICS ===
Analysis of 'begroting' debates from 2014 to 2024
Total unique normalized 'begroting' titles: 50
Total unique 'begroting' debate IDs: 371
Total 'begroting' speeches: 94532
Percentage of all debates: 2.09%
Percentage of all speeches: 11.93%

=== NORMALIZED 'BEGROTING' TITLES WITH AT LEAST ONE DEBATE EACH YEAR ===
Normalized Title                         | Debates  | Speeches | Orig. #  | 2014     | 2015     | 2016     | 2017     | 2018     | 2019     | 2020     | 2021     | 2022     | 2023     | 2024    
----------------------------------------------------------------------------------------------------------------------------------

Total topics with debates in every year: 0

=== NORMALIZED 'BEGROTING' TITLES WITH GAPS IN YEARLY DEBATES ===
Normalized Title                         | Debates  | Speeches | Orig. #  | 2014     | 2015     | 2016     | 2017     | 2018     | 2019     | 2020     | 2021     | 2022     | 2023     | 2024    
------------------